# L2 · Stage 1: teacher answers for the 7B → 3B distillation (GPU T4 ×2)

Task **L2**, step 1 of 2, from `Essential_task_delegation/Next_Steps_Roadmap.md`. Goal **O4**: a distilled 3B model.

This is a copy of `support/Abrar/Implementation_DisasterM3/generate_teacher_cot_train_3b_distill.ipynb`, which stays unchanged as the original. It runs the fine-tuned 7B teacher (`AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP`) over part of the training manifest. Its output, `disasterm3_distillation_cot_train.json`, feeds step 2, `train_qwen_disasterm3_3b_distill.ipynb`, through `DISTILL_MANIFEST_PATH`.

**What the teacher writes (smoke test, 19 Oct):** although the prompt asks it to think step by step, our fine-tuned 7B answers in one sentence, in the style of the training labels it learned. So this step produces **checked teacher answers**: answer-level distillation, not chain-of-thought. That suits the field model, which should answer briefly and fast. Explaining *why* is planned as a separate second skill (task **L2b** in the roadmap), with explanations grounded in facts from the masks.

**What changed from the original** (19 Oct 2026; neither version had run before):
- **No pinned installs.** The original pins torch 2.4.1, transformers 4.49 and bitsandbytes 0.45. They have no wheels for Kaggle's current Python 3.13, so its Cell 1 would ask for a kernel restart forever. This copy uses Kaggle's own torch, as H1 V2 does.
- **fp16 on both T4s, in batches of 8**, instead of 4-bit on one T4 and one example at a time. The smoke test took about 15 seconds per batch of 8, so 4,000 examples take about 2 hours. The original would have needed about 100 GPU-hours for its 12,000.
- **4,000 examples from the six tasks whose answer can be checked:** building and road counting, disaster type, bearing bodies, land use and relational reasoning. Captions and restoration advice are long free text with nothing to check, so they keep their plain labels.
- **Only correct answers are kept.** Cell 6 checks each answer's last line against the label:
  - counting: the same number (number words such as "three" count);
  - disaster type, bearing bodies and land use: the same categories, checked against **every category the task uses**, not just the question's 8 options, so an extra category the teacher adds (such as "roads") also counts as wrong;
  - relational reasoning: the labelled option is the closest one.

  Tested on `train_release.json`, 99.9–100% of the labels themselves pass, per task. A wrong answer is dropped, and step 2 trains that example on its plain label, so the 3B never learns the teacher's mistakes.

### How to run
1. Settings: **GPU T4 ×2**, internet ON, the DisasterM3 mirror attached (as for the training notebooks). Secret `HF_TOKEN` if the teacher repo is private; `KAGGLE_USERNAME` + `KAGGLE_KEY` only for the optional push in Cell 7.
2. Optional smoke test with `SMOKE_TEST = True`: 16 examples, about 15 minutes including the 16 GB model download. The three printed answers should answer the question in one sentence, and some should match their label.
3. With `SMOKE_TEST = False`, use Save & Run All (about 2 hours). The run stops cleanly at 11 hours. To continue, attach that version's output and run again: Cell 5 copies its `teacher_cot_progress.jsonl` in and skips what is done.
4. Outputs in `/kaggle/working/`:
   - `disasterm3_distillation_cot_train.json`: every eligible training entry, with `teacher_rationale` filled only where it was checked correct;
   - `teacher_cot_progress.jsonl`: every raw answer;
   - `cot_match_by_task.csv`: how many were kept, per task.

**Step 2** (`train_qwen_disasterm3_3b_distill.ipynb`) still has the same old pinned installs and needs the same environment fix before it can run.

In [1]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
# The original's pins (torch 2.4.1, transformers 4.49, bitsandbytes 0.45) have no wheels for Kaggle's
# Python 3.13, so this cell pins nothing. Same approach as h1-v2-counts-as-text.ipynb.
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B teacher in fp16 needs both T4s"

✓ torch 2.11.0+cu128, transformers 5.16.1, python 3.13.15
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os, json, random, re, time, glob, shutil, difflib
from pathlib import Path
from datetime import datetime

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"   # same original manifest the 7B notebook trained on

TEACHER_MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"

# Same image-token budget as the 7B training run -- the teacher was trained
# with this cap, so generation should stay consistent with it.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

# Generation sampling -- temperature 0.2 gives some rationale diversity
# without drifting off the ground-truth answer.
GEN_TEMPERATURE = 0.2
GEN_MAX_NEW_TOKENS = 320       # a short rationale plus the one-line answer (the original: 400)
DO_SAMPLE = True
BATCH = 8                      # examples per generate() call; halved automatically on out-of-memory

SMOKE_TEST = False             # True = 16 examples, to check the setup and read a few rationales
# How many training entries get a rationale. QLoRA's lesson: a smaller, cleaner set beats a large
# noisy one. Everything else is still written to the output manifest, with teacher_rationale empty,
# and Stage 2 falls back to the plain label for those.
SUBSET_SIZE = 16 if SMOKE_TEST else 4000
RANDOM_SEED = 42
# Tasks whose answer can be checked against the label (Cell 6). Captions and restoration advice are
# long free text with no single answer, so they keep their plain labels.
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}

OUTPUT_DIR = Path("/kaggle/working")
PROGRESS_PATH = OUTPUT_DIR / "teacher_cot_progress.jsonl"   # incremental, resumable
FINAL_MANIFEST_PATH = OUTPUT_DIR / "disasterm3_distillation_cot_train.json"
TIME_LIMIT_HOURS = 11.0  # stop cleanly before Kaggle's 12h wall (loading takes about 10 minutes of it)

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

print(f"Teacher: {TEACHER_MODEL_ID}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Subset size: {SUBSET_SIZE:,}{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
print(f"Devices: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

Teacher: AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP
Manifest: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_release.json
Subset size: 4,000
Devices: ['Tesla T4', 'Tesla T4']


In [3]:
# ── Cell 3: Load the teacher in fp16, split across both T4s ───────────────
# The original loaded it in 4-bit NF4 on one T4, which is slow and slightly changes the teacher's answers;
# fp16 over two T4s is the setup H1 V2 uses for this model.
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

print(f"Loading teacher {TEACHER_MODEL_ID} in fp16 across {torch.cuda.device_count()} GPUs...")
processor = AutoProcessor.from_pretrained(
    TEACHER_MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    token=hf_token(),
)
processor.tokenizer.padding_side = "left"              # batched generation needs left padding
# transformers 4.56 renamed torch_dtype to dtype; older versions may ignore dtype and load fp32 (30 GB)
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    TEACHER_MODEL_ID,
    **{dtype_arg: torch.float16},
    device_map="auto",
    attn_implementation="sdpa",
    token=hf_token(),
)
model.eval()

print(f"Teacher loaded ({dtype_arg}=float16); GPU memory "
      f"{[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")

Loading teacher AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP in fp16 across 2 GPUs...


preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Teacher loaded (dtype=float16); GPU memory ['7.6 GB', '9.0 GB']


In [4]:
# ── Cell 4: Load manifest, pick the subset, build the prompt, and the answer check ──
from collections import defaultdict
from PIL import Image

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from {MANIFEST_PATH}")

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

def resolve_image_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "box_train_images" / rel_path,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "box_train_images" / "box_train_images" / filename,
        DATA_ROOT / "masks" / "masks" / filename,
        Path("/tmp") / rel_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    # Same exclusions the 7B training notebook applied: drop segmentation /
    # path-like-answer entries and anything whose images don't resolve.
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

eligible_entries = [e for e in raw_data if eligible(e)]
print(f"Eligible for the distillation manifest: {len(eligible_entries):,} / {len(raw_data):,}")

# The subset is drawn from the tasks whose answer Cell 6 can check
checkable = [i for i, e in enumerate(eligible_entries) if e.get("task") in CHECKABLE]
random.seed(RANDOM_SEED)
subset_indices = set(random.sample(checkable, k=min(SUBSET_SIZE, len(checkable))))
print(f"Generating answers for {len(subset_indices):,} of the {len(checkable):,} checkable entries "
      f"(everything else keeps its plain label; Stage 2 falls back to it).")

COT_INSTRUCTION = (
    " Think step by step: describe what you observe in the pre- and "
    "post-disaster imagery, then give your final answer on its own line in "
    "the exact format of the target label."
)

def build_gen_messages(entry):
    """-> (chat messages, PIL images): the images first, then the training prompt plus the instruction.
    The processor resizes each image to the MIN_PIXELS..MAX_PIXELS budget, as in training."""
    keys = ["image_path"] if "image_path" in entry else ["pre_image_path", "post_image_path"]
    images = [Image.open(resolve_image_path(entry[k])).convert("RGB") for k in keys]
    prompt = entry.get("prompts", "")
    if isinstance(prompt, list):
        prompt = prompt[0]
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt + COT_INSTRUCTION}]
    return [{"role": "user", "content": content}], images

NUMBER = re.compile(r"\d+(?:\.\d+)?")
NUMBER_WORDS = {w: str(i) for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen "
    "seventeen eighteen nineteen twenty".split())}       # small counts are often written as words
GENERIC = {"area", "areas"}                              # 'Residential area' is also said 'residential zones'
ALIASES = {"volcano": ["volcanic"]}
CATEGORY_TASKS = ("Disaster Type Recognition", "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition")

def final_line(text):
    lines = [line.strip() for line in str(text).strip().splitlines() if line.strip()]
    return lines[-1] if lines else ""

def numbers(text):
    """The numbers in a text, with number words ('three') read as digits."""
    text = re.sub(r"\b(" + "|".join(NUMBER_WORDS) + r")\b", lambda m: NUMBER_WORDS[m.group(1)], str(text).lower())
    return {float(x) for x in NUMBER.findall(text)}

def canon(name):
    """One key per category: lower case, spaces collapsed, 'area(s)' dropped, a final 's' dropped,
    so 'Forest', 'Forests' and 'Forest  ' are the same category."""
    words = [w for w in str(name).lower().split() if w not in GENERIC]
    key = " ".join(words) or " ".join(str(name).lower().split())
    return key[:-1] if key.endswith("s") else key

def mentioned(option, text):
    """Is this option named in the text? Tolerates plurals and wordings like 'forested' or 'residential zones';
    whole words only, so 'port' is not found in 'airport', nor 'dams' in 'damaged'."""
    words = [w for w in str(option).lower().split() if w not in GENERIC] or str(option).lower().split()
    phrase = " ".join(words)
    phrase = phrase[:-1] if phrase.endswith("s") else phrase
    names = [phrase] + ALIASES.get(str(option).lower(), [])
    return any(re.search(rf"\b{re.escape(n)}(?:s|es|ed|ing)?\b", text) for n in names)

# Every category each task uses, across all its questions (a question shows only 8 of them). The dataset's
# option lists hold an empty name, 'None', stray notes in brackets and variants with spaces: those are dropped.
VOCAB = defaultdict(dict)                                # task -> {category key: one of its names}
for e in raw_data:
    if e.get("task") in CATEGORY_TASKS:
        for o in map(str, e.get("options_list") or []):
            key = canon(o)
            if key and key != "none" and "(" not in o:
                VOCAB[e["task"]].setdefault(key, o)
print(f"Categories per task: { {t: len(v) for t, v in VOCAB.items()} }")

def matches_label(entry, rationale):
    """True if the answer's last line gives the labelled answer: the same number for counting; for disaster type,
    bearing bodies and land use, exactly the labelled categories among every category the task uses (so an extra
    one counts as wrong); for relational reasoning, the labelled option as the closest one, at least 0.8 similar.
    Anything unclear counts as no match. On train_release.json, 99.9-100% of the labels themselves pass, per task
    (checked 19 Oct)."""
    answer, truth, task = final_line(rationale).lower(), entry.get("ground_truth"), entry.get("task")
    if task in ("Building Damage Counting", "Road Damage Counting"):
        return numbers(answer) == numbers(truth)
    if task == "relational reasoning":
        options = [str(o).lower().rstrip(".") for o in entry["options_list"]]
        score = {o: difflib.SequenceMatcher(None, answer.rstrip("."), o).ratio() for o in options}
        best = max(score, key=score.get)
        return best == str(truth).lower().rstrip(".") and score[best] >= 0.8
    if task in CATEGORY_TASKS:
        said = {key for key, name in VOCAB[task].items() if mentioned(name, answer)}
        return said == {canon(t) for t in str(truth).split(",") if canon(t)}
    return False

# Self-checks on label formats copied from train_release.json
_bdc = {"task": "Building Damage Counting", "ground_truth": 231, "options_list": [231, 323, 277, 139, 185]}
_bdc3 = {"task": "Building Damage Counting", "ground_truth": 3, "options_list": [3, 5, 7, 9, 11]}
_road = {"task": "Road Damage Counting", "ground_truth": "5.22%", "options_list": ["1.11%", "4.72%", "5.22%"]}
_dtr = {"task": "Disaster Type Recognition", "ground_truth": "explosion"}
_volcano = {"task": "Disaster Type Recognition", "ground_truth": "volcano"}
_bbr = {"task": "Disaster Bearing Bodies Recognition", "ground_truth": "Buildings, Roads, Forest, Open-space ground"}
_bbr2 = {"task": "Disaster Bearing Bodies Recognition", "ground_truth": "Farmland, Forest"}
_dsr = {"task": "Disaster Scene Recognition", "ground_truth": "Parking, Commercial area"}
_rel = {"task": "relational reasoning",
        "ground_truth": "The destroyed buildings in the red box are below the red intact building in the blue box.",
        "options_list": ["The destroyed buildings in the red box are below the red intact building in the blue box",
                         "The cluster of damaged buildings in the red box is situated directly below the intact "
                         "road in the blue box"]}
assert matches_label(_bdc, "I count the roofs.\nThe total number of undamaged buildings after the disaster is 231.")
assert not matches_label(_bdc, "The total number of undamaged buildings after the disaster is 230.")
assert matches_label(_bdc3, "A total of three buildings were completely destroyed in the disaster.")
assert matches_label(_road, "Roads are narrow.\nThe undamaged road area represents 5.22% of the total image area.")
assert matches_label(_dtr, "Collapsed roofs around a crater.\nThis area has been affected by an explosion.")
assert not matches_label(_dtr, "This area has been affected by a fire.")
assert matches_label(_volcano, "This region has been affected by a volcanic disaster.")
assert matches_label(_bbr, "The damaged objects include buildings, roads, forested areas, and open-space ground.")
assert not matches_label(_bbr, "The damaged objects include buildings and roads.")
assert not matches_label(_bbr2, "The critical objects that show disaster-related damage include farmland, forest, and roads.")
assert matches_label(_bbr2, "The critical objects that show disaster-related damage include farmland and forest.")
assert matches_label(_dsr, "The land-use zones present include parking areas and commercial zones.")
assert not matches_label(_dsr, "The land-use zones present include parking areas and residential zones.")
assert matches_label(_rel, "The destroyed buildings in the red box are below the red intact building in the blue box.")
assert not matches_label(_rel, "The cluster of damaged buildings in the red box is situated directly below the "
                               "intact road in the blue box.")
assert not matches_label({"task": "disaster caption", "ground_truth": "x"}, "x")
print("✓ Prompt and answer check ready (self-checks passed)")

Loaded 92,968 entries from /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_release.json
Eligible for the distillation manifest: 55,764 / 92,968
Generating answers for 4,000 of the 40,233 checkable entries (everything else keeps its plain label; Stage 2 falls back to it).
Categories per task: {'Disaster Bearing Bodies Recognition': 16, 'Disaster Type Recognition': 10, 'Disaster Scene Recognition': 17}
✓ Prompt and answer check ready (self-checks passed)


In [5]:
# ── Cell 5: Generation loop (batched, resumable, written as it goes) ───────
from tqdm.auto import tqdm

# Continue an earlier run: copy its progress in when that run's output is attached
previous = sorted(glob.glob("/kaggle/input/**/teacher_cot_progress.jsonl", recursive=True))
if previous and not PROGRESS_PATH.exists():
    shutil.copy(previous[0], PROGRESS_PATH)
    print(f"Continuing from {previous[0]}")

already_done = set()
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        already_done = {json.loads(line)["_idx"] for line in f if line.strip()}
    print(f"Resuming: {len(already_done):,} entries already generated in a prior session.")

@torch.inference_mode()
def generate(batch):
    """batch: list of (messages, images) -> list of rationales."""
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in batch]
    images = [img for _, imgs in batch for img in imgs]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=DO_SAMPLE,
                         temperature=GEN_TEMPERATURE)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(new_tokens, skip_special_tokens=True)]

def generate_safe(batch):
    """Halves the batch on out-of-memory instead of crashing the run."""
    try:
        return generate(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [""]
        half = len(batch) // 2
        return generate_safe(batch[:half]) + generate_safe(batch[half:])

todo = sorted(i for i in subset_indices if i not in already_done)
print(f"{len(todo):,} entries left to generate.")
deadline = time.time() + TIME_LIMIT_HOURS * 3600
t0, stopped, shown = time.time(), False, 0
with open(PROGRESS_PATH, "a", encoding="utf-8") as progress_fh:
    for start in tqdm(range(0, len(todo), BATCH), desc="Rationales"):
        if time.time() > deadline:
            stopped = True
            break
        idxs = todo[start:start + BATCH]
        rationales = generate_safe([build_gen_messages(eligible_entries[i]) for i in idxs])
        for i, rationale in zip(idxs, rationales):
            progress_fh.write(json.dumps({"_idx": i, "teacher_rationale": rationale}) + "\n")
            if shown < 3:                      # the first few, to read before trusting the run
                e = eligible_entries[i]
                print(f"\n[{e['task']}] label: {answer_text(e)[:120]!r}\n"
                      f"  rationale ({'matches' if matches_label(e, rationale) else 'does NOT match'} the label): "
                      f"{rationale[:400]!r}")
                shown += 1
        progress_fh.flush()

print(f"\n{'Time budget reached: stopped cleanly' if stopped else 'Generation pass complete'} after "
      f"{(time.time() - t0) / 3600:.1f} h this session.")

4,000 entries left to generate.


Rationales:   0%|          | 0/500 [00:00<?, ?it/s]


[Disaster Bearing Bodies Recognition] label: 'The essential land-cover objects that appear to be damaged in this disaster zone include buildings, roads, forested area'
  rationale (does NOT match the label): 'The essential land-cover objects that appear to have sustained damage in this disaster zone include buildings, roads, forest, and farmland.'

[Disaster Bearing Bodies Recognition] label: 'The critical objects that show disaster-related damage include buildings, roads, forest, farmland, and open-space ground'
  rationale (does NOT match the label): 'The critical objects that show disaster-related damage include buildings, roads, forest, and farmland.'

[Disaster Bearing Bodies Recognition] label: 'The key objects exhibiting visible impact from this disaster event include buildings, roads, forested areas, and open-sp'
  rationale (matches the label): 'The key objects that display visible impact from this disaster event include buildings, roads, forested areas, and open-space ground

In [6]:
# ── Cell 6: Keep only checked-correct rationales and write the distillation manifest ──
# Re-read the full progress log (covers this session + any prior ones) and merge teacher_rationale back
# onto every manifest entry (not just the subset), so the file is a drop-in replacement for
# train_release.json in the Stage-2 notebook. A rationale is kept only if its last line gives the
# labelled answer (matches_label, Cell 4); a wrong or missing one leaves teacher_rationale empty, and
# Stage 2 trains that entry on its plain label instead.
import pandas as pd

rationale_by_idx = {}
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            rationale_by_idx[row["_idx"]] = row["teacher_rationale"]

print(f"Total rationales generated so far: {len(rationale_by_idx):,} / {len(subset_indices):,} targeted")

distill_manifest, per_task = [], {}
for idx, entry in enumerate(eligible_entries):
    out_entry = dict(entry)
    rationale = rationale_by_idx.get(idx, "")
    correct = bool(rationale) and matches_label(entry, rationale)
    out_entry["teacher_rationale"] = rationale if correct else ""
    distill_manifest.append(out_entry)
    if rationale:
        counts = per_task.setdefault(entry["task"], {"generated": 0, "kept": 0})
        counts["generated"] += 1
        counts["kept"] += correct

by_task = pd.DataFrame([{"task": t, **c, "kept_%": round(100 * c["kept"] / c["generated"], 1)}
                        for t, c in sorted(per_task.items())])
by_task.to_csv(OUTPUT_DIR / "cot_match_by_task.csv", index=False)
print(by_task.to_string(index=False))

with open(FINAL_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(distill_manifest, f, ensure_ascii=False, indent=2)

n_kept = sum(1 for e in distill_manifest if e["teacher_rationale"])
print(f"\nWrote {FINAL_MANIFEST_PATH}")
print(f"  Entries total: {len(distill_manifest):,}")
print(f"  With a checked-correct teacher_rationale: {n_kept:,} of {len(rationale_by_idx):,} generated")
print(f"  Without (Stage 2 trains these on the plain label): {len(distill_manifest) - n_kept:,}")

if len(rationale_by_idx) < len(subset_indices):
    print(f"\n NOTE: generation is not finished ({len(rationale_by_idx):,}/{len(subset_indices):,} of the "
          f"targeted subset). Attach this version's output and run again: Cell 5 continues from "
          f"{PROGRESS_PATH.name}, and Cell 6 only needs its final pass at the end.")

Total rationales generated so far: 4,000 / 4,000 targeted
                               task  generated  kept  kept_%
           Building Damage Counting       1445   225    15.6
Disaster Bearing Bodies Recognition        776   331    42.7
         Disaster Scene Recognition        694   353    50.9
          Disaster Type Recognition        170   152    89.4
               Road Damage Counting        735     0     0.0
               relational reasoning        180    24    13.3

Wrote /kaggle/working/disasterm3_distillation_cot_train.json
  Entries total: 55,764
  With a checked-correct teacher_rationale: 1,085 of 4,000 generated
  Without (Stage 2 trains these on the plain label): 54,679


In [7]:
# ── Cell 7: Push the manifest to a Kaggle Dataset (optional) ─────────────
# So it can be attached as input to train_qwen_3b_distill.ipynb via
# /kaggle/input/disasterm3-teacher-cot/disasterm3_distillation_cot_train.json
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
    username = os.environ["KAGGLE_USERNAME"]

    ds_slug = f"{username}/disasterm3-teacher-cot"
    with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
        json.dump({
            "title": "disasterm3-teacher-cot",
            "id": ds_slug,
            "licenses": [{"name": "CC0-1.0"}],
        }, f)

    r = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if "already exists" in (r.stdout + r.stderr).lower():
        r = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR),
             "-m", f"cot manifest @ {datetime.now().isoformat()}", "--dir-mode", "zip"],
            capture_output=True, text=True,
        )
    print((r.stdout or r.stderr)[-2000:])
    print(f"Pushed to kaggle.com/datasets/{ds_slug}")
    print("Attach it as input to train_qwen_3b_distill.ipynb, then set:")
    print("  DISTILL_MANIFEST_PATH = Path('/kaggle/input/disasterm3-teacher-cot/disasterm3_distillation_cot_train.json')")
except Exception as e:
    print(f"Kaggle dataset push skipped ({e}).")
    print(f"The manifest is still available locally at {FINAL_MANIFEST_PATH} -- "
          f"download it manually, or run 'Save Version' to export /kaggle/working.")

Dataset creation error: The requested title "disasterm3-teacher-cot" is already in use by a dataset. Please choose another title.

Pushed to kaggle.com/datasets/abrarmohammedtanzim/disasterm3-teacher-cot
Attach it as input to train_qwen_3b_distill.ipynb, then set:
  DISTILL_MANIFEST_PATH = Path('/kaggle/input/disasterm3-teacher-cot/disasterm3_distillation_cot_train.json')
